# Phase 10 — Ablation: grounded vs ungrounded (quantitative)

Scores affected/not-affected classification on a labeled set under three conditions:
- **system** — deterministic version-aware rule (the KG logic; reference).
- **ungrounded** — the LLM from name+version+CVE only.
- **grounded** — the LLM given the deciding fact (fixed version) + the comparison rule.

The four decoys (patched versions that match by name) are where ungrounded should over-flag. LLM conditions need Ollama.

In [9]:
import sys, importlib; sys.path.append('../src')
import ablation; importlib.reload(ablation)   # reload in case the module changed
gt = ablation.load_gt('../eval/ground_truth.json')
print(f'{len(gt)} labeled pairs:',
      sum(e['affected'] for e in gt), 'affected /', sum(not e['affected'] for e in gt), 'not affected')

7 labeled pairs: 3 affected / 4 not affected


In [10]:
# SYSTEM (deterministic KG rule) — no LLM needed; should be perfect vs ground truth
_, m_sys = ablation.run(gt, ablation.classify_system)
print('system   :', m_sys)

system   : {'TP': 3, 'FP': 0, 'FN': 0, 'TN': 4, 'precision': 1.0, 'recall': 1.0, 'FPR': 0.0, 'F1': 1.0, 'accuracy': 1.0}


### Debug — see what the model actually says (raw answers)
Run this first if the metrics look wrong. It prints each pair's raw ungrounded/grounded reply and the parsed decision.

In [11]:
for r in ablation.debug(gt):
    t = 'AFF' if r['truth'] else 'not'
    print(f"{r['name']:<8} {r['version']:<7} {r['cve']:<16} truth={t}")
    print(f"   ungrounded -> {'AFF' if r['ung'] else 'not':<3} | {r['ung_raw']}")
    print(f"   grounded   -> {'AFF' if r['gnd'] else 'not':<3} | {r['gnd_raw']}")

openssh  9.6p1   CVE-2024-6387    truth=AFF
   ungrounded -> not | I've checked the relevant sources, but I couldn't find any information about a specific vulnerabilit
   grounded   -> AFF | To determine whether the installed software version (9.6p1) is affected by the vulnerability CVE-202
glibc    2.39    CVE-2024-2961    truth=AFF
   ungrounded -> not | I do not have access to real-time information about specific software versions and their vulnerabili
   grounded   -> AFF | To determine whether the installed software version (2.39) is affected by the vulnerability CVE-2024
curl     8.7.1   CVE-2024-7264    truth=AFF
   ungrounded -> not | I do not have access to the latest information about the specific software package "curl" version "8
   grounded   -> AFF | To determine whether the installed software version (8.7.1) is affected by the vulnerability CVE-202
zlib     1.3.1   CVE-2022-37434   truth=not
   ungrounded -> not | After checking, I found that the vulnerability CVE-2022-3

In [12]:
# LLM conditions (requires Ollama + llama3.1:8b)
_, m_ung = ablation.run(gt, ablation.classify_ungrounded)
_, m_gnd = ablation.run(gt, ablation.classify_grounded)
print('ungrounded:', m_ung)
print('grounded  :', m_gnd)

ungrounded: {'TP': 0, 'FP': 0, 'FN': 3, 'TN': 4, 'precision': 0.0, 'recall': 0.0, 'FPR': 0.0, 'F1': 0.0, 'accuracy': 0.571}
grounded  : {'TP': 3, 'FP': 0, 'FN': 0, 'TN': 4, 'precision': 1.0, 'recall': 1.0, 'FPR': 0.0, 'F1': 1.0, 'accuracy': 1.0}


In [13]:
# Results table
cols = ['precision','recall','FPR','F1','accuracy','FP','FN']
print(f"{'condition':<11}" + ''.join(f'{c:>10}' for c in cols))
for name, m in [('system', m_sys), ('ungrounded', m_ung), ('grounded', m_gnd)]:
    print(f'{name:<11}' + ''.join(f'{m[c]:>10}' for c in cols))

condition   precision    recall       FPR        F1  accuracy        FP        FN
system            1.0       1.0       0.0       1.0       1.0         0         0
ungrounded        0.0       0.0       0.0       0.0     0.571         0         3
grounded          1.0       1.0       0.0       1.0       1.0         0         0


## Reading the result
- **system** should be perfect (precision/recall = 1, FPR = 0): the KG version-aware rule is correct by construction.
- **ungrounded** should show low recall and/or a high false-positive rate — it doesn't reliably know fixed versions.
- **grounded** should recover most/all of that gap once the fixed-version fact + rule are supplied.

If the two LLM rows look identical or all-zero, run the **debug** cell above to see the raw answers — usually the model is refusing to give a clean verdict, which the prompt now forces via `VERDICT: ...`.

The **FPR / recall gap from ungrounded → grounded** is the headline number for the paper. Expand `ground_truth.json` and report mean ± CI over runs + a paired McNemar test for the final paper.